In [ ]:
import os, re, sys, json, shutil, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch

sys.path.append('../..')
sys.path.append('../../Model')    # O Blocks.index CARREGA A REDE PELO Network/ DE Model/
from Blocks.index import BlockPredictor, BlockSection, FaultStickExtractor, TileGrid, getBlockInfo, getSeismic, getVolume, writeTiles, modelSummary, saveReport, saveViewer

cv2.setNumThreads(1)              # O DATALOADER CRIA WORKERS POR FORK DEPOIS DO CV2: SEM ISSO OS FILHOS TRAVAM
pd.set_option('display.max_columns', None)

In [ ]:
{'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}

# RECORTE DO ARTIGO
- O F3_wu é o subvolume do F3 que o Wu et al. (2019) usam na figura 7 e no `gxl.dat` do repositório deles: 128 (tempo) × 384 × 512 amostras, que no nosso F3 é **inline 0:512, z 334:462, xline 0:384** (a análise de 20/11/2026 bateu o `gxl.dat` com o bloco)
- O dado sai dos próprios tiles de `Blocks/F3` (mesma escala p01/p99 em `[0, 1]`): o volume do F3 é montado pelos núcleos (`getVolume`), recortado e cortado de novo em tiles de 128³ com 32 voxels de borda pelo `writeTiles`, no mesmo formato do gerador dos tiles (borda espelhada); regerar os tiles do F3 inteiro pelo `writeTiles` dá os mesmos `.dat`, com diferença zero
- O recorte é um bloco por si: as redes só veem o recorte, como no artigo (a borda de cima, em z 334, é espelhada e não o dado de cima)
- Só grava quando o `patch_metadata.json` ainda não existe; a prova é o volume montado pelos tiles novos igual ao recorte

In [ ]:
SOURCE_META = '../F3/files/patch_metadata.json'
SOURCE_DIR  = '../F3/files/patches'
CROP        = (slice(0, 512), slice(334, 462), slice(0, 384))    # (inline, z, xline) DO gxl.dat DO WU

source = getVolume(TileGrid(SOURCE_META), SOURCE_DIR, '../F3/files/cache/volume.npy')
crop   = np.array(source[CROP])

if not os.path.exists('files/patch_metadata.json'):
    writeTiles(crop, 'files/patches', {'crop_origin': [CROP[0].start, CROP[1].start, CROP[2].start], 'full_shape': list(source.shape), 'norm_method': 'normalize', 'percentile': 99.0, 'source': 'Blocks/F3, recorte do gxl.dat do Wu'}, metaPath='files/patch_metadata.json')

volume = getVolume(TileGrid('files/patch_metadata.json'), 'files/patches', 'files/cache/volume.npy')
{'recorte': crop.shape, 'volume dos tiles': volume.shape, 'igual ao recorte': bool(np.array_equal(volume, crop)), 'media': round(float(volume.mean()), 4), 'desvio': round(float(volume.std()), 4)}

# CONFIGURAÇÃO
- O F3_wu chega como o F3: tiles de 128³ com 32 voxels de borda, float32 em `[0, 1]`, eixos `(inline, z, xline)`; o bloco é 512 × 128 × 384 em 8 × 2 × 6 = 96 tiles
- `BASE_PATH` é a pasta dos modelos (`model_N/` com `info.json`); `MODELS = []` pega todos os `model_N` dela. As referências (`Documents/Reference/projeto_*/results`) têm o `predicted.npy` pronto, gravado pelo código dos autores, e o notebook só analisa
- `INLINE = 29` e `TIME_SLICE = 99` (z 433) são as fatias que o `prediction.ipynb` do Wu desenha; a fatia de tempo é a vista principal da figura 7 do artigo
- `SAVE_PRED = True` grava o `predicted.npy` (512 × 128 × 384, ~0,1 GB), que a fatia de tempo e o 3D leem; `SKIP_DONE = True` reaproveita o que já está em disco
- `STICK_MIN` e `STICK_GAP` são menores que os do Marlim porque a seção do recorte tem só 128 amostras de altura: com 140 px nenhuma falha caberia

In [ ]:
BLOCK      = 'F3_wu'
BASE_PATH  = '../../Documents/Backups/Marcia'    # PASTA DOS MODELOS: '../../Model/Backup', '../../Documents/Backups/Marcia', ...
TILES_DIR  = 'files/patches'                           # O BLOCO INTEIRO, UM .DAT POR TILE
META_PATH  = 'files/patch_metadata.json'
CACHE_DIR  = 'files/cache'
OUTPUT_DIR = f'{BASE_PATH}/outputs/{BLOCK.lower()}'    # FIGURAS E TABELA DE TODOS OS MODELOS SELECIONADOS

MODELS          = ['model_1']   # [] = TODOS OS MODELOS DE BASE_PATH
INLINE          = 29            # INLINE DAS FIGURAS (0 A 511); 29 É A DO prediction.ipynb DO WU
TIME_SLICE      = 99            # FATIA DE TEMPO DA FIGURA DO ARTIGO (0 A 127; z 433 NO F3)
FULL_PREDICT    = True          # False PREDIZ SÓ OS TILES QUE COBREM A INLINE; True O BLOCO INTEIRO
SAVE_TILES      = False         # True GRAVA A MÁSCARA DE CADA TILE EM masks/ (UM .DAT DE ~8 MB POR TILE)
SAVE_PRED       = True          # True GRAVA O predicted.npy DO BLOCO (~0,1 GB, PEDE FULL_PREDICT); A FATIA DE TEMPO E O 3D LEEM DELE
SKIP_DONE       = True          # True REAPROVEITA O QUE JÁ ESTÁ EM DISCO (predicted.npy OU masks/)
OVERLAP         = 0.25          # SOBREPOSIÇÃO ENTRE JANELAS (0 = CORTE SECO, DO JEITO QUE O FORMAT CORTA O TREINO)
WINDOW          = None          # None = A JANELA DO TREINO (info.json); FORCE UMA TUPLA PARA TESTAR OUTRA, EX: (64, 64, 64)
MODEL_THRESHOLD = 0.5           # PROBABILIDADE A PARTIR DA QUAL O VOXEL ENTRA NA MÁSCARA

STICK_TOL = 0.80    # PERMISSIVIDADE DO EXTRATOR: 0 SÓ O TRAÇO ÓBVIO, 1 TUDO QUE PARECER FALHA
STICK_MIN = 60      # PX: A SEÇÃO TEM SÓ 128 AMOSTRAS DE ALTURA (OS 140 DO MARLIM NÃO CABEM NELA)
STICK_GAP = 25      # PX: MAIOR LACUNA COSTURADA DENTRO DE UM STICK (OS 59 DO MARLIM NA MESMA PROPORÇÃO)

models = MODELS or sorted((m for m in os.listdir(BASE_PATH) if re.fullmatch(r'model_\d+', m)), key=lambda n: int(n.split('_')[-1]))
models

# BLOCO
- `TileGrid` lê o `patch_metadata.json` e sabe onde o núcleo de cada tile (o tile sem os 32 voxels de borda) cai no volume, no mesmo corte em que os tiles foram gerados; o volume é montado pelos núcleos, com o shape do metadata, e um tile faltando levanta erro em vez de deixar buraco zerado
- `getBlockInfo` resume o bloco: shape inteiro, tamanho do tile, grade, total de tiles, quantos faltam em disco e o mínimo, máximo, média e desvio dos valores numa amostra de 100 tiles espalhados pela grade
- Tile normalizado (min-max ou p01/p99) fica em $[0, 1]$; padronizado fica com média 0 e desvio 1

In [ ]:
grid = TileGrid(META_PATH)
getBlockInfo(grid, TILES_DIR)

# MODELOS
- Rede, janela de treino, dataset e IoU de teste de cada modelo, lidos do `info.json`; o `rotulo` vai na legenda das figuras

In [ ]:
pd.DataFrame([modelSummary(BASE_PATH, name) for name in models])

# PREDIÇÃO
- `BlockPredictor` cuida de um modelo: carrega os pesos só quando algum tile está pendente, e só se o `scaling` do treino for `'normalize'` ou `'percentile'` (o bloco está em `[0, 1]`)
- Cada tile passa pela `SlidingWindow` na janela do treino: as janelas se somam com peso de Hanning (~1 no centro, ~0 na borda) para a emenda não virar degrau, e o tile menor que a janela é espelhado
- O núcleo de cada tile predito vai direto para a inline das figuras e, com `SAVE_PRED`, para o `predicted.npy`; a tabela abaixo é quantos tiles a rodada cobre e quantos ainda faltam predizer

In [ ]:
options    = {'overlap': OVERLAP, 'window': WINDOW, 'skipDone': SKIP_DONE, 'saveTiles': SAVE_TILES, 'savePred': SAVE_PRED}
predictors = {name: BlockPredictor(f'{BASE_PATH}/{name}', **options) for name in models}
region     = None if FULL_PREDICT else INLINE    # None = O BLOCO INTEIRO
pd.DataFrame([{'modelo': name, 'tiles': len(grid.names(region)), 'pendentes': len(predictors[name].pending(grid, f'{BASE_PATH}/{name}/{BLOCK.lower()}', region))} for name in models])

# EXTRAÇÃO DE STICKS
- Cada falha de uma seção 2D vira uma reta (**stick**), a mesma representação para a rede e para o especialista
- **Evidência**: a probabilidade suavizada e reescalada para $[0,1]$; o piso sobe pelo percentil quando a predição é densa, para a teia não explodir o traçado
- **Votação tipo Hough**: a evidência somada numa janela reta de `seedLen` px em cada orientação plausível; os picos são as sementes
- **Traçado**: a semente se estende enquanto houver suporte, tolerando lacunas de até `STICK_GAP` px, e a reta é reajustada ao centro de massa da evidência por **PCA**
- **Filtros**: mergulho entre 43° e 76°, comprimento mínimo `STICK_MIN`, preenchimento, **energia sísmica** (a falha precisa cortar refletores, não a água nem a zona morta) e **ângulo com as camadas** pelo tensor de estrutura
- **Matching pursuit**: aceita a melhor reta, consome o corredor dela e segue para a próxima; no fim, retas colineares se fundem e as que correm por cima de uma maior saem

In [ ]:
extractor = FaultStickExtractor(STICK_TOL, STICK_MIN, STICK_GAP)
pd.Series({'tolerance': extractor.tolerance, 'minLen': extractor.minLen, 'maxGap': extractor.maxGap, 'dipMin': extractor.dipMin, 'dipMax': extractor.dipMax})

# SEÇÃO DO BLOCO
- `BlockSection` faz o papel do `FaultComparer` do Marlim: recebe a inline predita, extrai os sticks e desenha os painéis; o F3 não tem anotação, então não há métrica contra o especialista
- A sísmica da `INLINE` sai só dos tiles que a cobrem (2 × 6 tiles), sem montar o bloco inteiro, e fica em `files/cache/real_inline_<inline>.npy`

In [ ]:
seismic = getSeismic(grid, TILES_DIR, f'{CACHE_DIR}/real_inline_{INLINE}.npy', INLINE)
section = BlockSection(seismic, extractor, MODEL_THRESHOLD)
section.show('seismic', title=f'{BLOCK} · inline {INLINE}')

# PREDIÇÃO E FIGURAS POR MODELO
- Para cada modelo: prediz os tiles pendentes, extrai os sticks da inline e grava as três figuras na pasta `f3_wu/` do modelo (`predicted_seismic_<inline>`, `predicted_mask_<inline>`, `predicted_sticks_<inline>`), copiadas para `OUTPUT_DIR` com o modelo e a rede no nome
- O `predicted.npy` é montado direto em disco por memmap e só ganha o nome final quando fica completo, então um arquivo interrompido nunca passa por pronto
- Um modelo sem `model.pth`, ou com `scaling` fora de `[0, 1]`, fica fora da tabela; a rede é solta antes do próximo modelo

In [ ]:
# PREDIZ O BLOCO COM UM MODELO, SALVA AS TRÊS FIGURAS DA INLINE (NA PASTA DO MODELO E EM OUTPUT_DIR) E DEVOLVE A LINHA DA TABELA
def predictedBlock(name):
    model    = modelSummary(BASE_PATH, name)
    blockDir = f'{BASE_PATH}/{name}/{BLOCK.lower()}'
    values   = predictors[name].predict(grid, TILES_DIR, blockDir, INLINE, FULL_PREDICT)

    if values is None:
        return None

    section = BlockSection(seismic, extractor, MODEL_THRESHOLD)
    section.setSection(model['rotulo'], values, 'red')

    metrics = section.metrics(model['rotulo'])
    title   = f"{BLOCK} (Inline {INLINE}) | Dataset: {str(model['dataset']).upper()} | Modelo: {name} ({model['rede']})"
    titles  = {'seismic': title, 'mask': title, 'sticks': f"{title} | Sticks: {metrics['sticks']}"}

    for kind, text in titles.items():
        section.show(kind, title=text, save=f'{blockDir}/predicted_{kind}_{INLINE}.png')
        shutil.copy(f'{blockDir}/predicted_{kind}_{INLINE}.png', f'{OUTPUT_DIR}/{name}_{model["rede"]}_inline_{INLINE}_{kind}.png')

    run = json.load(open(f'{blockDir}/predict.json')) if os.path.exists(f'{blockDir}/predict.json') else {}
    return {'modelo': name, **{key: model[key] for key in ('rede', 'janela', 'iou_teste')}, 'overlap': run.get('overlap'), 'inline': INLINE, **metrics}


os.makedirs(OUTPUT_DIR, exist_ok=True)
rows = []

for name in models:
    rows.append(predictedBlock(name))
    predictors[name].reset()

rows = [row for row in rows if row]
pd.DataFrame(rows)

# FATIA DE TEMPO DO ARTIGO
- A vista principal da figura 7 do Wu: a fatia de tempo `TIME_SLICE` com a predição em vermelho onde a probabilidade passa do `MODEL_THRESHOLD`; à direita, a mesma predição no fundo preto
- Grava `predicted_timeslice_<z>.png` na pasta `f3_wu/` do modelo e a cópia em `OUTPUT_DIR`; a tabela é a fração da fatia acima do limiar

In [ ]:
# A FATIA DE TEMPO DE UM MODELO NOS DOIS FUNDOS (SÍSMICA E PRETO), COMO A FIGURA DO ARTIGO; DEVOLVE A FRAÇÃO DA FATIA ACIMA DO LIMIAR
def timeSlice(name):
    blockDir = f'{BASE_PATH}/{name}/{BLOCK.lower()}'
    if not os.path.exists(f'{blockDir}/predicted.npy'):
        return None

    model  = modelSummary(BASE_PATH, name)
    probs  = np.array(np.load(f'{blockDir}/predicted.npy', mmap_mode='r')[:, TIME_SLICE, :]).T
    seis   = np.array(volume[:, TIME_SLICE, :]).T
    masked = np.ma.masked_less(probs, MODEL_THRESHOLD)    # A PREDIÇÃO EM VERMELHO, COMO AS OUTRAS FIGURAS DOS BLOCOS
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    for ax, background, label in zip(axes, (seis, np.zeros_like(seis)), ('sísmica', 'fundo preto')):
        ax.imshow(background, cmap='gray', aspect='auto')
        ax.imshow(masked, cmap=ListedColormap(['red']), aspect='auto')
        ax.set_title(f"{BLOCK} · z {TIME_SLICE + CROP[1].start} · {name} ({model['rede']}) · {label} · threshold {MODEL_THRESHOLD}")
        ax.set_xlabel('inline'); ax.set_ylabel('crossline')

    fig.savefig(f'{blockDir}/predicted_timeslice_{TIME_SLICE}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    shutil.copy(f'{blockDir}/predicted_timeslice_{TIME_SLICE}.png', f'{OUTPUT_DIR}/{name}_{model["rede"]}_timeslice_{TIME_SLICE}.png')
    return {'modelo': name, 'rede': model['rede'], 'fatia': TIME_SLICE, 'fracao_fatia': round(float((probs >= MODEL_THRESHOLD).mean()), 4)}


pd.DataFrame([row for row in map(timeSlice, models) if row])

# VISUALIZAÇÃO 3D
- Um `view3d.html` por modelo na pasta `f3_wu/` dele, como a figura 3D do Wu: fatia de tempo e as duas paredes do fundo (inline e crossline) cortadas nela, com a probabilidade em jet por cima
- No HTML: limiar ajustável, sísmica + predição ou só a predição no fundo preto, posição das fatias, exagero vertical, voxels 3D opcionais e a ficha da rede (receita, dataset, IoU de teste, como o bloco foi predito)
- Tudo em resolução cheia: a predição vai inteira no HTML (uint8, piso 0,05, o menor limiar dele) e a sísmica do recorte é gravada uma vez em `files/cache/view3d/` e lida pelo HTML por caminho relativo

In [ ]:
start = (0, TIME_SLICE, 0)    # PAREDES INLINE 0 E CROSSLINE 0, FATIA DE TEMPO DO ARTIGO
for name in models:
    print(saveViewer(BASE_PATH, name, f'{BASE_PATH}/{name}/{BLOCK.lower()}', volume, BLOCK, f'{CACHE_DIR}/view3d', MODEL_THRESHOLD, (CROP[0].start, CROP[1].start, CROP[2].start), start=start))

# TABELA NA RAIZ
- `sticks_report.csv` fica em `OUTPUT_DIR` e junta as linhas novas às do CSV anterior: um par (modelo, inline) recalculado substitui o antigo, e rede, janela e IoU de teste são relidos do `info.json`

In [ ]:
saveReport(rows, f'{OUTPUT_DIR}/sticks_report.csv', BASE_PATH, 'inline', 'sticks')